# Stage 2: Capital Allocator (CLARE)

This notebook implements Stage 2 of CLARE:

1. Train an allocator model to estimate safe lending capacity.
2. Integrate with Stage 1 PD for risk-adjusted limits.

Core haircut logic:

$$
Final\_Limit = Raw\_Capacity \times (1 - PD)
$$

Where:

- $PD = P(Default \mid x)$ from Stage 1 classifier
- $Raw\_Capacity$ is the Stage 2 regressor output

In [1]:
import warnings
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = Path("accepted_loans.csv")
PLOT_DIR = Path("plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = ARTIFACT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_TRIALS_STAGE2 = 20
MAX_ROWS_FOR_TUNING = 250_000
MAX_ROWS_STAGE1_TRAIN = 600_000

BAD_STATUSES = {"Charged Off", "Default", "Late (31-120 days)"}
GOOD_STATUSES = {"Fully Paid", "Current"}
TARGET_STATUSES = BAD_STATUSES.union(GOOD_STATUSES)

LEAKAGE_DROP_COLS = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "hardship_flag",
    "debt_settlement_flag",
]

ADMIN_DROP_COLS = ["id", "member_id", "url", "desc", "title", "zip_code"]
PRIORITY_FEATURES = ["annual_inc", "dti", "emp_length", "total_rev_hi_lim"]

STAGE1_MODEL_PATH = Path("stage1_classifier.json")
STAGE2_MODEL_PATH = Path("stage2_regressor.json")
STAGE1_MODEL_ARTIFACT_PATH = MODEL_DIR / "stage1_classifier.json"
STAGE2_MODEL_ARTIFACT_PATH = MODEL_DIR / "stage2_regressor.json"
STAGE1_BUNDLE_PATH = MODEL_DIR / "stage1_preprocessor_bundle.joblib"
STAGE2_BUNDLE_PATH = MODEL_DIR / "stage2_preprocessor_bundle.joblib"

c:\Users\Ashvik\projects\CLARE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def save_current_plot(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def parse_int_rate(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = series.astype("string").str.replace("%", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


def optimize_memory(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=["int64", "int32"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")

    object_cols = df.select_dtypes(include=["object"]).columns
    for col in object_cols:
        unique_ratio = df[col].nunique(dropna=False) / max(len(df), 1)
        if unique_ratio < 0.2:
            df[col] = df[col].astype("category")
    return df


def load_full_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=True)
    df = optimize_memory(df)

    if "loan_status" in df.columns:
        df["loan_status"] = df["loan_status"].astype("string").str.strip()
    if "issue_d" in df.columns:
        df["issue_year"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce").dt.year
    if "int_rate" in df.columns:
        df["int_rate"] = parse_int_rate(df["int_rate"])

    for numeric_col in ["dti", "annual_inc", "total_rev_hi_lim", "loan_amnt"]:
        if numeric_col in df.columns:
            df[numeric_col] = pd.to_numeric(df[numeric_col], errors="coerce")

    return df


def apply_leakage_and_admin_drop(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = [c for c in LEAKAGE_DROP_COLS + ADMIN_DROP_COLS if c in df.columns]
    return df.drop(columns=drop_cols)


def build_preprocessor_bundle(df: pd.DataFrame, target_col: str, priority_features=None):
    if priority_features is None:
        priority_features = []

    feature_df = df.drop(columns=[target_col]).copy()

    missing_ratio = feature_df.isna().mean()
    drop_cols = missing_ratio[missing_ratio > 0.90].index.tolist()
    flag_cols = missing_ratio[(missing_ratio >= 0.50) & (missing_ratio <= 0.90)].index.tolist()

    work_df = feature_df.drop(columns=drop_cols, errors="ignore").copy()

    for col in flag_cols:
        if col in work_df.columns:
            work_df[f"{col}_is_missing"] = work_df[col].isna().astype("int8")

    numeric_cols = work_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in work_df.columns if c not in numeric_cols]

    num_imputer = SimpleImputer(strategy="median")
    if numeric_cols:
        work_df[numeric_cols] = num_imputer.fit_transform(work_df[numeric_cols])

    encoders = {}
    for col in categorical_cols:
        series = work_df[col].astype("string")
        mode_vals = series.dropna().mode()
        fill_value = mode_vals.iloc[0] if not mode_vals.empty else "MISSING"
        filled = series.fillna(fill_value)

        encoder = LabelEncoder()
        encoder.fit(filled)
        mapping = {cls_val: int(idx) for idx, cls_val in enumerate(encoder.classes_)}

        work_df[col] = filled.map(mapping).astype("int32")
        encoders[col] = {
            "encoder": encoder,
            "fill_value": fill_value,
            "mapping": mapping,
        }

    scaler = RobustScaler()
    if numeric_cols:
        work_df[numeric_cols] = scaler.fit_transform(work_df[numeric_cols])

    feature_cols = work_df.columns.tolist()
    prioritized = [c for c in priority_features if c in feature_cols]
    remaining = [c for c in feature_cols if c not in prioritized]
    feature_cols = prioritized + remaining

    work_df = work_df[feature_cols]

    bundle = {
        "drop_cols": drop_cols,
        "flag_cols": flag_cols,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "num_imputer": num_imputer,
        "encoders": encoders,
        "scaler": scaler,
        "feature_cols": feature_cols,
        "target_col": target_col,
    }

    return work_df, bundle


def transform_with_bundle(df: pd.DataFrame, bundle: dict, target_col: str = None) -> pd.DataFrame:
    work_df = df.copy()

    if target_col and target_col in work_df.columns:
        work_df = work_df.drop(columns=[target_col])

    work_df = work_df.drop(columns=bundle["drop_cols"], errors="ignore")

    for col in bundle["flag_cols"]:
        flag_name = f"{col}_is_missing"
        if col in work_df.columns:
            work_df[flag_name] = work_df[col].isna().astype("int8")
        else:
            work_df[flag_name] = 1

    for col in bundle["numeric_cols"]:
        if col not in work_df.columns:
            work_df[col] = np.nan

    for col in bundle["categorical_cols"]:
        if col not in work_df.columns:
            work_df[col] = pd.NA

    if bundle["numeric_cols"]:
        work_df[bundle["numeric_cols"]] = bundle["num_imputer"].transform(work_df[bundle["numeric_cols"]])

    for col in bundle["categorical_cols"]:
        info = bundle["encoders"][col]
        fill_value = info["fill_value"]
        mapping = info["mapping"]

        encoded = work_df[col].astype("string").fillna(fill_value).map(mapping)
        work_df[col] = encoded.fillna(-1).astype("int32")

    if bundle["numeric_cols"]:
        work_df[bundle["numeric_cols"]] = bundle["scaler"].transform(work_df[bundle["numeric_cols"]])

    for col in bundle["feature_cols"]:
        if col not in work_df.columns:
            work_df[col] = 0

    work_df = work_df[bundle["feature_cols"]]
    return work_df

In [3]:
full_df = load_full_data(DATA_PATH)

stage1_df = full_df[full_df["loan_status"].isin(TARGET_STATUSES)].copy()
stage1_df["default_flag"] = stage1_df["loan_status"].isin(BAD_STATUSES).astype("int8")
stage1_df = apply_leakage_and_admin_drop(stage1_df)

stage2_df = full_df[full_df["loan_status"] == "Fully Paid"].copy()
stage2_df = stage2_df.dropna(subset=["loan_amnt"])
stage2_df = apply_leakage_and_admin_drop(stage2_df)

print(f"Total rows loaded: {len(full_df):,}")
print(f"Stage 1 population rows: {len(stage1_df):,}")
print(f"Stage 2 population rows (Fully Paid): {len(stage2_df):,}")

available_priority = [c for c in PRIORITY_FEATURES if c in stage2_df.columns]
missing_priority = [c for c in PRIORITY_FEATURES if c not in stage2_df.columns]
print(f"Priority features available: {available_priority}")
print(f"Priority features missing: {missing_priority}")

Total rows loaded: 2,260,701
Stage 1 population rows: 2,245,134
Stage 2 population rows (Fully Paid): 1,076,751
Priority features available: ['annual_inc', 'dti', 'emp_length', 'total_rev_hi_lim']
Priority features missing: []


In [4]:
def train_or_load_stage1(stage1_frame: pd.DataFrame):
    if STAGE1_MODEL_PATH.exists() and STAGE1_BUNDLE_PATH.exists():
        model = XGBClassifier()
        model.load_model(str(STAGE1_MODEL_PATH))
        bundle = joblib.load(STAGE1_BUNDLE_PATH)
        return model, bundle, {"status": "loaded"}

    train_frame = stage1_frame.copy().drop(columns=["loan_status"], errors="ignore")

    if len(train_frame) > MAX_ROWS_STAGE1_TRAIN:
        train_frame, _ = train_test_split(
            train_frame,
            train_size=MAX_ROWS_STAGE1_TRAIN,
            stratify=train_frame["default_flag"],
            random_state=RANDOM_STATE,
        )

    X_all, bundle = build_preprocessor_bundle(
        train_frame,
        target_col="default_flag",
        priority_features=PRIORITY_FEATURES,
    )
    y_all = train_frame["default_flag"].astype(int).values

    X_train, X_val, y_train, y_val = train_test_split(
        X_all,
        y_all,
        test_size=0.20,
        stratify=y_all,
        random_state=RANDOM_STATE,
    )

    n_good = int((y_train == 0).sum())
    n_bad = int((y_train == 1).sum())
    scale_pos_weight = n_good / max(n_bad, 1)

    model = XGBClassifier(
        n_estimators=450,
        max_depth=6,
        learning_rate=0.05,
        gamma=2.0,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
    )

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_pd = model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_pd)

    model.save_model(str(STAGE1_MODEL_PATH))
    model.save_model(str(STAGE1_MODEL_ARTIFACT_PATH))
    joblib.dump(bundle, STAGE1_BUNDLE_PATH)

    return model, bundle, {
        "status": "trained",
        "validation_auc": float(val_auc),
        "rows_used": int(len(train_frame)),
    }


stage1_model, stage1_bundle, stage1_meta = train_or_load_stage1(stage1_df)
print(stage1_meta)

{'status': 'trained', 'validation_auc': 0.7665391747605568, 'rows_used': 600000}


In [6]:
def fit_optuna_stage2_regressor(X_train: pd.DataFrame, y_train: np.ndarray):
    if len(X_train) > MAX_ROWS_FOR_TUNING:
        X_tune, _, y_tune, _ = train_test_split(
            X_train,
            y_train,
            train_size=MAX_ROWS_FOR_TUNING,
            random_state=RANDOM_STATE,
        )
    else:
        X_tune, y_tune = X_train, y_train

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_tune,
        y_tune,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 8.0),
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "objective": "reg:squarederror",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "tree_method": "hist",
        }

        model = XGBRegressor(**params)
        model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
        pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        return rmse

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS_STAGE2, show_progress_bar=False)

    best = study.best_trial.params
    best_params = {
        "n_estimators": best["n_estimators"],
        "max_depth": best["max_depth"],
        "learning_rate": best["learning_rate"],
        "gamma": best["gamma"],
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "objective": "reg:squarederror",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    model = XGBRegressor(**best_params)
    model.fit(X_train, y_train, verbose=False)

    return model, best_params, study, X_tune, y_tune


stage2_model_df = stage2_df.copy().drop(columns=["loan_status"], errors="ignore")
stage2_train_df, stage2_test_df = train_test_split(
    stage2_model_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

X_stage2_train, stage2_bundle = build_preprocessor_bundle(
    stage2_train_df,
    target_col="loan_amnt",
    priority_features=PRIORITY_FEATURES,
)
y_stage2_train = stage2_train_df["loan_amnt"].astype(float).values

X_stage2_test = transform_with_bundle(stage2_test_df, stage2_bundle, target_col="loan_amnt")
y_stage2_test = stage2_test_df["loan_amnt"].astype(float).values

stage2_model, stage2_best_params, stage2_study, X_stage2_tune, y_stage2_tune = fit_optuna_stage2_regressor(
    X_stage2_train,
    y_stage2_train,
)

stage2_pred = stage2_model.predict(X_stage2_test)
stage2_test_mae = mean_absolute_error(y_stage2_test, stage2_pred)
stage2_test_rmse = np.sqrt(mean_squared_error(y_stage2_test, stage2_pred))
stage2_test_r2 = r2_score(y_stage2_test, stage2_pred)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_model = XGBRegressor(**stage2_best_params)
cv_mae_scores = -cross_val_score(cv_model, X_stage2_tune, y_stage2_tune, scoring="neg_mean_absolute_error", cv=kf, n_jobs=1)
cv_rmse_scores = -cross_val_score(cv_model, X_stage2_tune, y_stage2_tune, scoring="neg_root_mean_squared_error", cv=kf, n_jobs=1)

stage2_model.save_model(str(STAGE2_MODEL_PATH))
stage2_model.save_model(str(STAGE2_MODEL_ARTIFACT_PATH))
joblib.dump(stage2_bundle, STAGE2_BUNDLE_PATH)

print(f"Stage 2 Test MAE: {stage2_test_mae:,.2f}")
print(f"Stage 2 Test RMSE: {stage2_test_rmse:,.2f}")
print(f"Stage 2 Test R2: {stage2_test_r2:.4f}")
print(f"Stage 2 CV MAE: {np.round(cv_mae_scores, 2)} | mean={cv_mae_scores.mean():,.2f}")
print(f"Stage 2 CV RMSE: {np.round(cv_rmse_scores, 2)} | mean={cv_rmse_scores.mean():,.2f}")
print(f"Best Optuna RMSE (validation): {stage2_study.best_value:,.2f}")

Stage 2 Test MAE: 46.62
Stage 2 Test RMSE: 235.37
Stage 2 Test R2: 0.9993
Stage 2 CV MAE: [48.97 47.   46.57 47.41 48.52] | mean=47.69
Stage 2 CV RMSE: [242.14 238.89 235.07 218.99 217.34] | mean=230.49
Best Optuna RMSE (validation): 240.35


In [7]:
plt.figure(figsize=(9, 7))
sns.scatterplot(x=y_stage2_test, y=stage2_pred, alpha=0.25, s=18)
line_min = min(float(np.min(y_stage2_test)), float(np.min(stage2_pred)))
line_max = max(float(np.max(y_stage2_test)), float(np.max(stage2_pred)))
plt.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="red", linewidth=2)
plt.title("Stage 2 Parity Plot: Predicted vs Actual Loan Amount")
plt.xlabel("Actual Loan Amount")
plt.ylabel("Predicted Loan Amount")
save_current_plot(PLOT_DIR / "stage2_parity_plot.png")

residuals = y_stage2_test - stage2_pred
plt.figure(figsize=(10, 6))
sns.scatterplot(x=stage2_pred, y=residuals, alpha=0.25, s=18)
plt.axhline(0, linestyle="--", color="red", linewidth=2)
plt.title("Stage 2 Residual Plot")
plt.xlabel("Predicted Loan Amount")
plt.ylabel("Residual (Actual - Predicted)")
save_current_plot(PLOT_DIR / "stage2_residual_plot.png")

X_stage1_on_stage2_test = transform_with_bundle(stage2_test_df, stage1_bundle, target_col="default_flag")
pd_scores_test = stage1_model.predict_proba(X_stage1_on_stage2_test)[:, 1]
raw_capacity_test = np.clip(stage2_pred, 0, None)
final_limit_test = raw_capacity_test * (1 - pd_scores_test)
avg_risk_haircut_pct = float(np.mean(pd_scores_test) * 100)

rng = np.random.default_rng(RANDOM_STATE)
sim_n = min(1000, len(stage2_test_df))
sim_idx = rng.choice(len(stage2_test_df), size=sim_n, replace=False)

simulation_df = pd.DataFrame(
    {
        "PD": pd_scores_test[sim_idx],
        "Raw_Capacity": raw_capacity_test[sim_idx],
        "Final_Limit": final_limit_test[sim_idx],
        "Risk_Haircut_Pct": pd_scores_test[sim_idx] * 100,
    }
)

plt.figure(figsize=(10, 6))
sns.histplot(simulation_df["Final_Limit"], bins=45, kde=True)
plt.title("Final Limit Distribution (1,000 Sample Simulation)")
plt.xlabel("Final Risk-Adjusted Limit")
plt.ylabel("Frequency")
save_current_plot(PLOT_DIR / "final_limit_distribution_simulation.png")

simulation_df.to_csv(ARTIFACT_DIR / "stage2_final_limit_simulation.csv", index=False)

print(f"Average Risk Haircut (%): {avg_risk_haircut_pct:.2f}")
print(f"Simulation rows: {len(simulation_df):,}")

Average Risk Haircut (%): 45.29
Simulation rows: 1,000


In [8]:
def get_risk_adjusted_limit(applicant_data):
    stage1_loaded = XGBClassifier()
    stage1_loaded.load_model(str(STAGE1_MODEL_PATH))

    stage2_loaded = XGBRegressor()
    stage2_loaded.load_model(str(STAGE2_MODEL_PATH))

    stage1_bundle_loaded = joblib.load(STAGE1_BUNDLE_PATH)
    stage2_bundle_loaded = joblib.load(STAGE2_BUNDLE_PATH)

    if isinstance(applicant_data, dict):
        applicant_df = pd.DataFrame([applicant_data])
    elif isinstance(applicant_data, pd.Series):
        applicant_df = applicant_data.to_frame().T
    else:
        applicant_df = applicant_data.copy()

    x_stage1 = transform_with_bundle(applicant_df, stage1_bundle_loaded, target_col="default_flag")
    x_stage2 = transform_with_bundle(applicant_df, stage2_bundle_loaded, target_col="loan_amnt")

    pd_prob = stage1_loaded.predict_proba(x_stage1)[:, 1]
    raw_capacity = np.clip(stage2_loaded.predict(x_stage2), 0, None)
    final_limit = raw_capacity * (1 - pd_prob)

    result_df = pd.DataFrame(
        {
            "PD": pd_prob,
            "Raw_Capacity": raw_capacity,
            "Final_Limit": final_limit,
            "Risk_Haircut_Pct": pd_prob * 100,
        }
    )
    return result_df


stage2_log = {
    "stage2_test_mae": float(stage2_test_mae),
    "stage2_test_rmse": float(stage2_test_rmse),
    "stage2_test_r2": float(stage2_test_r2),
    "stage2_cv_mae_mean": float(cv_mae_scores.mean()),
    "stage2_cv_rmse_mean": float(cv_rmse_scores.mean()),
    "average_risk_haircut_pct": float(avg_risk_haircut_pct),
    "stage2_model_path": str(STAGE2_MODEL_PATH.resolve()),
}

with open(ARTIFACT_DIR / "stage2_deployment_log.json", "w", encoding="utf-8") as f:
    json.dump(stage2_log, f, indent=2)

pd.DataFrame(
    {
        "metric": [
            "stage2_test_mae",
            "stage2_test_rmse",
            "stage2_test_r2",
            "stage2_cv_mae_mean",
            "stage2_cv_rmse_mean",
            "average_risk_haircut_pct",
        ],
        "value": [
            float(stage2_test_mae),
            float(stage2_test_rmse),
            float(stage2_test_r2),
            float(cv_mae_scores.mean()),
            float(cv_rmse_scores.mean()),
            float(avg_risk_haircut_pct),
        ],
    }
).to_csv(ARTIFACT_DIR / "stage2_allocator_metrics.csv", index=False)

print(f"Stage 2 model saved to: {STAGE2_MODEL_PATH.resolve()}")
print(f"Deployment log saved to: {(ARTIFACT_DIR / 'stage2_deployment_log.json').resolve()}")
print(f"Average Risk Haircut (%): {avg_risk_haircut_pct:.2f}")

example_applicants = stage2_test_df.drop(columns=["loan_amnt"], errors="ignore").sample(3, random_state=RANDOM_STATE)
example_limits = get_risk_adjusted_limit(example_applicants)
example_limits

Stage 2 model saved to: C:\Users\Ashvik\projects\CLARE\stage2_regressor.json
Deployment log saved to: C:\Users\Ashvik\projects\CLARE\artifacts\stage2_deployment_log.json
Average Risk Haircut (%): 45.29


,PD,Raw_Capacity,Final_Limit,Risk_Haircut_Pct
0,0.440598,2006.846069,1122.632812,44.059841
1,0.424575,7988.677734,4596.887207,42.457470
2,0.682570,19964.458984,6337.312500,68.257027
